# Phase D — GRPO training

Trains the adaptive policy with TRL `GRPOTrainer` + LoRA. **Runtime → Change runtime type → GPU.**

### Run order (from the proposal)

1. **Smoke test** — 5 steps, tiny group. Answers only "does it execute".
2. **Go/no-go gate** — `--lambda-think 0.0`, i.e. correctness + format, no length penalty.
   If GRPO cannot improve plain correctness over base, the adaptive question is moot and the
   project pivots to characterising why.
3. **λ sweep** — only after the gate passes.

### What to watch

`metric_think_rate` in the logs, not the reward curve. Collapse to all-think or all-no-think
is the primary failure mode and it is invisible in reward alone — a policy that stopped
reasoning and one that started emitting malformed calls both flatten it.

From the fp16 baselines: the **oracle thinks on 17.4%** of items, prompting alone gives 96.9%.
A trained policy near 17–20% is in the right regime; near 0% or near 100% is collapse.

### Why λ is swept where it is

Break-even λ per category, computed from the fp16 baselines: `simple_python` never worth
thinking (Δ = −2.8%), `multiple` 0.07, `parallel` 0.48, `parallel_multiple` 0.53,
`irrelevance` 2.11. Four of five switch off below 0.55, then nothing changes until 2.11 —
so `{0.05, 0.1, 0.25, 0.5, 1.0, 2.0}` covers every transition and a linear sweep would not.

## 1 — Install

Separate from the generation notebook's environment. Training needs `trl`/`peft`/`datasets`;
it does **not** need vLLM, so we do not repeat that CUDA fight here.

`bfcl-eval` is `--no-deps` for the same reason as before: only its data files are used, and
its full dependency tree pins an incompatible torch.

**If Colab offers RESTART SESSION when this finishes, take it, then resume at step 2.**

In [ ]:
!pip install -q trl peft datasets accelerate
!pip install -q --no-deps bfcl-eval==2026.3.23

# Remove Colab's preinstalled torchao. Nothing here uses it — LoRA runs in fp16
# — but PEFT's LoRA dispatcher calls is_torchao_available() while deciding which
# module type to inject, and that helper *raises* on a too-old version instead of
# returning False:
#     ImportError: Found an incompatible version of torchao.
#     Found version 0.10.0, but only versions above 0.16.0 are supported
# Colab ships 0.10.0. Absent, the same helper returns False cleanly and the
# dispatcher moves on. Uninstalling is preferred over upgrading: it is
# deterministic and cannot drag a new dependency into a torch install that is
# already delicately balanced.
!pip uninstall -y -q torchao

## 2 — Verify, then get the code

In [ ]:
import importlib.util
import os

import bfcl_eval
import peft
import torch
import trl

data_dir = os.path.join(os.path.dirname(bfcl_eval.__file__), "data")
print("torch     :", torch.__version__, "| CUDA", torch.version.cuda)
print("trl       :", trl.__version__)
print("peft      :", peft.__version__)
print("bfcl data :", os.path.isdir(data_dir))
print("bf16      :", torch.cuda.is_bf16_supported(), "(False on T4 — fp16 fallback)")

# PEFT's LoRA dispatcher raises rather than skipping when an old torchao is
# present, so its absence is a precondition, not a preference.
print("torchao   :", "absent (good)" if importlib.util.find_spec("torchao") is None else "PRESENT — rerun the uninstall")

assert os.path.isdir(data_dir), "BFCL data missing — rerun the --no-deps install"
assert importlib.util.find_spec("torchao") is None, "torchao still installed; PEFT will raise on LoRA injection"
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
REPO = "https://github.com/widodu77/rs_aidams.git"

if os.path.isdir("/content/rs_aidams"):
    !cd /content/rs_aidams && git pull --ff-only
else:
    !git clone -q $REPO /content/rs_aidams

%cd /content/rs_aidams
os.environ["PYTHONPATH"] = "src"
!git log --oneline -1

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_RUNS = "/content/drive/MyDrive/rs_aidams/runs"
os.makedirs(DRIVE_RUNS, exist_ok=True)

## 3 — Smoke test

Five steps. The only question is whether the loop executes end to end: dataset builds, prompts
render, rollouts sample, the reward function returns the right shape, LoRA attaches, a step
lands. Read nothing scientific into the numbers.

In [ ]:
!python -m train.train_grpo \
    --output runs/smoke \
    --lambda-think 0.0 \
    --max-steps 5 \
    --num-generations 4 \
    --per-device-batch-size 4 \
    --gradient-accumulation-steps 1 \
    --max-completion-length 256

## 4 — Go/no-go gate

`--lambda-think 0.0`: correctness + format, no length penalty. The question is whether GRPO
improves plain correctness over the base model at all.

Baseline to beat, on the **eval split only** (the split manifest is written into the run
directory so the baselines can be restricted to the same items):
adaptive-prompt = 87.5% accuracy at 287.5 mean tokens.

In [ ]:
!python -m train.train_grpo --output runs/gate --lambda-think 0.0 --max-steps 300
!cp -r runs/gate $DRIVE_RUNS/

In [ ]:
# Think-rate and correctness traces. Watch for collapse, not for a rising reward.
import json

history = json.load(open("runs/gate/log_history.json", encoding="utf-8"))
keys = ["step", "reward", "rewards/metric_think_rate/mean",
        "rewards/metric_correctness/mean", "rewards/metric_format_rate/mean"]
rows = [r for r in history if "reward" in r]
print(f"{'step':>6s} {'reward':>8s} {'think':>8s} {'correct':>8s} {'format':>8s}")
for r in rows[:: max(1, len(rows) // 25)]:
    vals = [r.get(k) for k in keys]
    print("".join(f"{(f'{v:.3f}' if isinstance(v, float) else str(v)):>9s}" for v in vals))
print("\noracle think-rate is 17.4%; near 0% or near 100% is collapse")

## 5 — λ sweep (only after the gate passes)

Each run is independent, so a session drop costs one λ rather than the sweep. Adapters are
copied to Drive as they finish.

In [ ]:
for lam in [0.05, 0.1, 0.25, 0.5, 1.0, 2.0]:
    tag = f"lam{lam}".replace(".", "_")
    if os.path.exists(f"{DRIVE_RUNS}/{tag}/adapter_model.safetensors"):
        print(f"skipping {tag}, already done")
        continue
    print(f"\n{'='*70}\nlambda = {lam}\n{'='*70}", flush=True)
    !python -m train.train_grpo --output runs/$tag --lambda-think $lam --max-steps 300
    !cp -r runs/$tag $DRIVE_RUNS/

## 6 — Evaluate a trained policy

Through `generate.run_vllm --adapter`, i.e. the **same** path that produced the baselines.
Scoring a trained policy through a different pipeline than its baselines is how comparisons
quietly break. Needs vLLM, so either run this in the generation notebook's session or install
it here (see `colab/generate_baselines.ipynb` for the CUDA 13 / torch cu130 fix).

In [ ]:
# !python -m generate.run_vllm --policy adaptive --adapter runs/gate \
#     --out results/raw/vllm/qwen3-1.7b_trained_gate.jsonl

## Then, locally

Score on CPU, restricted to the eval split recorded in `runs/<name>/split_manifest.json`:

```bash
uv run python -m analysis.score_run results/raw/vllm/qwen3-1.7b_trained_gate.jsonl
```